Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Free Energy Calculation with MBAR (including the additional sampling)

This notebook is an extended version of `06_mbar_free_energy_en.ipynb`.
The strongly-restrained windows (`KAPPA=10.0 eV`) added around RC~12 A in
`07_umbrella_sampling_extra_near_rc12_en.ipynb` are analyzed with MBAR together with the original windows (`KAPPA=2.5 eV`).

MBAR (Multistate Bennett Acceptance Ratio) correctly handles a different bias (spring constant and restraint center) for each window,
so windows with different spring constants can be combined rigorously.

## Step 0. Import libraries

In [ ]:
# Install the mbar package if needed
#! pip install pymbar

In [ ]:
# Reference
#   https://weitsehsu.com/course/enhanced_sampling/us/
#   https://github.com/choderalab/pymbar-examples/blob/master/umbrella-sampling-pmf/umbrella-sampling.py

import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import pymbar
from pymbar import timeseries

%config InlineBackend.figure_format = "retina"

## Step 1. Set the constants

The temperature is common to the original and the additional calculations (375 K). We precompute the inverse temperature $\beta=1/(k_BT)$.

In [ ]:
# Step1: Settings
kB   = 8.617333262e-5  # Boltzmann constant in eV/K
T    = 375.0           # single temperature (Kelvin) : common to original and extra
beta = 1 / (kB * T)
print(f"beta = {beta:.4f} 1/eV")

## Step 2. Define the windows to analyze (original + extra)

The original and the extra windows are registered as sets of center, spring constant, and output directory.
`build_window_list` scans each output directory and builds a list of `(center, kappa, dir)`.

- Original windows: `KAPPA=2.5 eV`
- Extra windows: `KAPPA=10.0 eV`
  (if a `meta.txt` exists, the `kappa` recorded there takes precedence)

**note**: If the directory `output/05_umbrella_sampling/` is not found, the original windows are loaded from `assets/05_umbrella_sampling/`. The extra windows require the output of 07 (`output/07_umbrella_sampling_extra/`).

In [ ]:
# ============================================================
# 2. Window definition (original + extra)
# ============================================================
KAPPA_ORIG  = 2.5    # eV : original US windows
KAPPA_EXTRA = 10.0   # eV : extra US windows (value used in 07; change it if you used another one)

# Load from output, fall back to assets if not found
ORIG_DIR  = "./output/05_umbrella_sampling"
if not os.path.isdir(ORIG_DIR):
    ORIG_DIR = "./assets/05_umbrella_sampling"
    print(f"Directory not found in output, loading from assets: {ORIG_DIR}")
EXTRA_DIR = "./output/07_umbrella_sampling_extra"  # output of 07 (not provided in assets)

def build_window_list(base_dir, default_kappa):
    """Scan cv_at_* under base_dir and return a list of (center, kappa, dir).
    If a kappa is written in meta.txt, it takes precedence."""
    windows = []
    for d in sorted(glob.glob(f"{base_dir}/cv_at_*")):
        center = float(os.path.basename(d).replace("cv_at_", ""))
        colvar = f"{d}/COLVAR_{center:.2f}"
        if not os.path.exists(colvar):
            continue  # skip windows that have not been computed
        kappa = default_kappa
        meta = f"{d}/meta.txt"
        if os.path.exists(meta):
            for line in open(meta):
                if line.startswith("kappa"):
                    kappa = float(line.split()[1])
        windows.append({"center": center, "kappa": kappa, "dir": d})
    return windows

windows = build_window_list(ORIG_DIR, KAPPA_ORIG) + build_window_list(EXTRA_DIR, KAPPA_EXTRA)
# Sort by center (for readability; the MBAR result does not depend on the order)
windows = sorted(windows, key=lambda w: (w["center"], w["kappa"]))

print(f"Total windows: {len(windows)}")
print(f"  original (kappa={KAPPA_ORIG}) : {sum(1 for w in windows if w['kappa']==KAPPA_ORIG)}")
print(f"  extra    (kappa={KAPPA_EXTRA}): {sum(1 for w in windows if w['kappa']==KAPPA_EXTRA)}")
for w in windows:
    print(f"  center={w['center']:6.2f}  kappa={w['kappa']:5.1f}  {w['dir']}")

## Step 3. Load the data

The following files are read from each window directory.

- `COLVAR_*` : time series of the reaction coordinate (z position) and the bias
- `md-dyn.log` : time series of the potential energy

To align the time stamps of the reaction coordinate and the potential energy, both are stored with the first entry (t=0) removed
(this resolves the length mismatch present in `06`). So that windows with different numbers of samples can be handled,
the data are stored in Python lists here.

In [ ]:
# ============================================================
# 3. Load trajectory data (variable-length safe)
# ============================================================
z_list = []   # reaction coordinate
w_list = []   # bias potential (for reference)
U_list = []   # potential energy (unbiased)
K_k    = []   # spring constant of each window
z0_k   = []   # restraint center of each window
beta_k = []   # inverse temperature of each window

for w in windows:
    center, kappa, d = w["center"], w["kappa"], w["dir"]

    colvar = np.loadtxt(f"{d}/COLVAR_{center:.2f}")

    with open(f"{d}/md-dyn.log") as f:
        log_lines = [line for line in f if "Time" not in line]
    pot = np.loadtxt(log_lines)[:, 2]  # potential energy [eV]

    # Align the time stamps of the reaction coordinate and the potential energy (drop t=0 from both)
    n = min(colvar.shape[0], pot.shape[0]) - 1  # common length (first entry excluded)
    z_list.append(colvar[1:1 + n, 1])
    w_list.append(colvar[1:1 + n, 2])
    U_list.append(pot[1:1 + n])

    K_k.append(kappa)
    z0_k.append(center)
    beta_k.append(beta)

K_k    = np.array(K_k)
z0_k   = np.array(z0_k)
beta_k = np.array(beta_k)

print(f"windows={len(windows)}, sample lengths: "
      f"min={min(len(z) for z in z_list)}, max={max(len(z) for z in z_list)}")

## Step 4. Data preprocessing (subsampling)

MD data are correlated in time, so we evaluate the statistical inefficiency with `pymbar.timeseries`
and extract only the decorrelated, independent samples. The number of samples `N_k` differs per window.

In [ ]:
# ============================================================
# 4. Subsampling (decorrelation)
# ============================================================
K     = len(windows)                       # number of windows
N_max = max(len(z) for z in z_list)         # maximum number of snapshots

N_k  = np.zeros([K], np.int32)
g_k  = np.zeros([K], np.float64)
z_kn = np.zeros([K, N_max], np.float64)
U_kn = np.zeros([K, N_max], np.float64)

for k in range(K):
    z_full = z_list[k]
    U_full = U_list[k]

    g_k[k]  = timeseries.statistical_inefficiency(z_full)
    indices = timeseries.subsample_correlated_data(z_full, g=g_k[k])
    N_k[k]  = len(indices)

    z_kn[k, :N_k[k]] = z_full[indices]
    U_kn[k, :N_k[k]] = U_full[indices]

print("Statistical inefficiency g_k (min/mean/max):",
      f"{g_k.min():.1f} / {g_k.mean():.1f} / {g_k.max():.1f}")
print("Independent samples N_k (min/mean/max):",
      f"{N_k.min()} / {N_k.mean():.0f} / {N_k.max()}")

## Step 5. Compute the reduced potential matrix

We compute the reduced potential obtained when snapshot $n$ of window $k$ is evaluated under the bias of another window $l$:

$$u_{k,l,n} = \beta\left[U(x_{kn}) + \tfrac{1}{2}\kappa_l (z_{kn}-z^0_l)^2\right]$$

The key point is that a **different $\kappa_l$** is used for each window.

In [ ]:
# ============================================================
# 5. Reduced potential matrix
# ============================================================
# Bin settings for the PMF plot
z_min = 10.0
z_max = 16.0
nbins = 60   # finer than the original (40) because the extra windows improved the resolution

bin_edges    = np.linspace(z_min, z_max, nbins + 1)
bin_center_i = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# u_kln[k,l,n] : reduced potential of snapshot n of window k evaluated under the bias of window l
N_max = int(np.max(N_k))
u_kln = np.zeros([K, K, N_max])

for k in range(K):
    for n in range(N_k[k]):
        # unbiased reduced potential
        base = beta_k[k] * U_kn[k, n]
        # bias of each window l (per-window kappa_l, z0_l)
        u_kln[k, :, n] = base + beta_k[k] * 0.5 * K_k * (z_kn[k, n] - z0_k) ** 2

# Unbiased reduced potential (for the FES evaluation)
u_kn = np.zeros([K, N_max])
for k in range(K):
    u_kn[k, :N_k[k]] = beta_k[k] * U_kn[k, :N_k[k]]

print("u_kln shape:", u_kln.shape)

## Step 6. Compute the free energy profile with MBAR

We reweight with `pymbar.FES` and obtain the unbiased PMF in histogram form.

In [ ]:
# ============================================================
# 6. Compute FES with MBAR
# ============================================================
fes = pymbar.FES(u_kln, N_k, verbose=True)
z_n = pymbar.utils.kn_to_n(z_kn, N_k=N_k)

histo_params = {"bin_edges": bin_edges}
fes.generate_fes(u_kn, z_n, fes_type="histogram", histogram_parameters=histo_params)
results = fes.get_fes(bin_center_i, reference_point="from-lowest",
                      uncertainty_method="analytical")

f_i  = results["f_i"]
df_i = results["df_i"]

out_dir = "./output/08_mbar_free_energy_with_extra"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/fes_with_extra.dat"
with open(out_path, "w") as f:
    f.write("# free energy profile (in units of kT), original + extra windows\n")
    f.write(f"# {'bin':>8s} {'f':>8s} {'df':>8s} \n")
    for i in range(len(bin_center_i)):
        f.write(f"{bin_center_i[i]:>8.3f} {f_i[i]:>8.3f} {df_i[i]:>8.3f} \n")
print("Saved:", out_path)

## Step 7. Visualize the results (before vs. after the additional sampling)

We plot the PMF including the additional sampling. If the result of the original `06` notebook
is still available, it is overlaid for comparison.

In [ ]:
# ============================================================
# 7. Plot: before vs after additional sampling
# ============================================================
data_new = np.loadtxt(out_path)
zc_new = data_new[:, 0]
f_new  = data_new[:, 1] - data_new[:, 1].min()
df_new = data_new[:, 2]

fig, ax = plt.subplots(figsize=(7, 5))

# Before the additional sampling (if available)
fes_orig = "./output/06_mbar_free_energy/fes.dat"
if os.path.exists(fes_orig):
    data_old = np.loadtxt(fes_orig)
    zc_old = data_old[:, 0]
    f_old  = data_old[:, 1] - data_old[:, 1].min()
    df_old = data_old[:, 2]
    ax.plot(zc_old, f_old, color="#8d99ae", linewidth=2.0,
            linestyle="--", label="Original only (06)")
    ax.fill_between(zc_old, f_old - df_old, f_old + df_old,
                    color="#8d99ae", alpha=0.20)

# After the additional sampling
ax.plot(zc_new, f_new, color="#023e8a", linewidth=2.5,
        label="Original + extra (08)")
ax.fill_between(zc_new, f_new - df_new, f_new + df_new,
                color="#ade8f4", alpha=0.8)

# Highlight the region covered by the additional sampling
ax.axvspan(11.6, 12.4, color="#ffd166", alpha=0.15, label="Extra sampling region")

ax.set_xlim(10, 16)
ax.set_ylim(0, 100)
ax.set_xlabel("Z position (A)", fontsize=14)
ax.set_ylabel("Free energy (kT)", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(fontsize=11)
ax.grid(True)

plt.savefig(f"{out_dir}/mbar_free_energy_with_extra.png", bbox_inches="tight")
plt.savefig(f"{out_dir}/mbar_free_energy_with_extra.pdf", bbox_inches="tight")
plt.show()

## Step 8. Check the overlap and the uncertainty near 12 A

To quantify the effect of the additional sampling, we check (1) the histogram overlap of each window and
(2) the statistical uncertainty `df_i` around 12 A.

In [ ]:
# Check the overlap using the histogram of each window (original = dashed, extra = solid)
stops = [(0.00, "#3366CC"), (0.5, "#E64B35"), (1.00, "#52C569")]
cmap = LinearSegmentedColormap.from_list("brg", stops)

fig, ax = plt.subplots(figsize=(8, 4))
for k, wdw in enumerate(windows):
    z = z_kn[k, :N_k[k]]
    hist, bins = np.histogram(z, bins=12, density=True)
    ctr = 0.5 * (bins[:-1] + bins[1:])
    ls = "--" if wdw["kappa"] == KAPPA_ORIG else "-"
    lw = 1.0 if wdw["kappa"] == KAPPA_ORIG else 1.8
    ax.plot(ctr, hist, ls, lw=lw, color=cmap(k / len(windows)), alpha=0.8)
ax.axvspan(11.6, 12.4, color="#ffd166", alpha=0.15)
ax.set_xlim(10, 16)
ax.set_xlabel("Reaction coordinate z (A)")
ax.set_ylabel("Probability density")
ax.set_title("Window overlap (dashed=original k=2.5, solid=extra k=10)")
ax.grid(True)
plt.show()

# Uncertainty near 12 A
mask = (zc_new >= 11.6) & (zc_new <= 12.4)
print(f"Mean statistical uncertainty df near 12 A (11.6-12.4): {df_new[mask].mean():.3f} kT")